credits: @Jan Majewski
https://github.com/Jan-Majewski/WWSI-GenAI/blob/main/notebooks/W1-tokenization-and-vectorization.ipynb

In [91]:
import pandas as pd
import numpy as np
import pickle
import openai
import os

from dotenv import load_dotenv
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from sklearn.neighbors import NearestNeighbors

In [92]:
word_embeddings = pickle.load(open("data/word_embeddings_subset.p", "rb"))
word_embeddings.keys()

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Missing OPENAI_API_KEY in .env")

C:\Users\rrble\AppData\Local\Temp\ipykernel_26556\1391670364.py:1: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  word_embeddings = pickle.load(open("data/word_embeddings_subset.p", "rb"))


# Wektorizacja
Utwórz funkcję, która pobiera tekst i model osadzania openai i zwraca osadzenie.

In [93]:
def get_embeddings(texts: list, model: str = "text-embedding-ada-002"):
    """Fetches embeddings for a list of texts using OpenAI API in a single batch call."""
    if not texts:
        return []
    try:
        response = openai.embeddings.create(
            model=model,
            input=texts
        )
        return [item.embedding for item in response.data]
    except Exception as e:
        print(f"Error getting embeddings: {e}")
        return [None] * len(texts)

def get_embedding(text: str, model: str = "text-embedding-ada-002"):
    """Fetches embedding for a single text. Provided for backward compatibility."""
    res = get_embeddings([text], model)
    return res[0] if res else None


In [94]:
df = pd.DataFrame(word_embeddings.keys(), columns=["word"])
embedding_model = "text-embedding-ada-002"

## Vectorize words in df using batching
words_list = df.word.tolist()
df["vector"] = get_embeddings(words_list, model=embedding_model)

## Drop words that failed to embed
df = df.dropna(subset=["vector"])
df["vector"] = df["vector"].apply(np.array)
print(f"Vectorized {len(df)} words.")

Vectorized 243 words.


## Redukcja wymiarowości – TSNE
Chcielibyśmy zobaczyć te wektory w przestrzeni 3D, więc przeprowadźmy eksperyment z TSNE.

In [95]:
def reduce_dimensions_tsne(embeddings, n_components=3, perplexity=20, random_state=42):
    """
    Reduces high-dimensional embeddings using t-SNE.

    Parameters:
    - embeddings (numpy.ndarray): High-dimensional embeddings.
    - n_components (int): Target number of dimensions (3 for 3D visualization).
    - perplexity (int): Controls balance between local and global aspects.
    - random_state (int): Ensures reproducibility.

    Returns:
    - numpy.ndarray: Reduced embeddings (N samples, 3D).
    """
    tsne = TSNE(n_components=n_components, perplexity=perplexity, random_state=random_state)
    return tsne.fit_transform(embeddings)


# Stack embeddings for t-SNE transformation
full_vectors = np.vstack(df["vector"].values)  # Change column as needed

# Reduce dimensionality
reduced_vectors = reduce_dimensions_tsne(full_vectors)
df["reduced_vec"] = reduced_vectors.tolist()
df_vectors_3d = pd.DataFrame(reduced_vectors, columns=["x", "y", "z"])
print(df["reduced_vec"][0])

df = df.merge(df_vectors_3d, left_index=True, right_index=True)

[-49.09098815917969, -0.3982391953468323, -19.02278709411621]


Sprawdź, ile osadzeń słów znajduje się w naszej próbce.

In [96]:
df.to_parquet("data/embeddings_test.parquet")
df = pd.read_parquet("data/embeddings_test.parquet")
df[["x", "y", "z"]].describe()

,x,y,z
count,243.000000,243.000000,243.000000
mean,-0.250861,0.272486,0.070087
std,27.334370,28.908794,23.689974
min,-61.819675,-58.379002,-52.803532
25%,-19.742167,-23.300544,-16.793579
50%,0.314233,0.297828,-0.101703
75%,19.548439,21.221443,15.837769
max,56.381031,63.475655,53.983250


Użyj dostarczonego szkieletu, aby utworzyć wykres 3D.

In [97]:
df = pd.read_parquet("data/embeddings_test.parquet")
df.rename(columns={"word": "label"}, inplace=True)
RANGE = 50

trace0 = go.Scatter3d(
    x=df.x,
    y=df.y,
    z=df.z,
    mode="markers",
    text=df.label
)

data = [trace0]

figure = go.Figure(
    data=data,
    layout=go.Layout(

        scene=dict(
            xaxis=dict(title="x", range=(-RANGE, RANGE)),
            yaxis=dict(title="y", range=(-RANGE, RANGE)),
            zaxis=dict(title="z", range=(-RANGE, RANGE))
        ),

    ))

name = ''

camera = dict(
    up=dict(x=0, y=0, z=1),
    center=dict(x=0, y=0, z=0),
    eye=dict(x=0, y=-1, z=1)
)

figure.update_layout(scene_camera=camera, title=name)  #

figure.show()

# Pokaż wektory krajów i stolic
Zbadajmy relacje między zestawem par kraj-stolica.

In [98]:
country_capitals_paris = [("France", "Paris"), ("England", "London"), ("Mali", "Bamako"), ("Italy", "Rome"),
                          ("Poland", "Warsaw"), ("Spain", "Madrid"), ("Kenya", "Nairobi"), ("Germany", "Berlin"),
                          ("Japan", "Tokyo"), ("China", "Beijing"), ('Jordan', 'Amman')]

## Prepare pairs (subsets of df) of country and capital vectors

data = []
for country, capital in country_capitals_paris:
    df_pair = df.loc[df.label.isin([country, capital])]

    trace = go.Scatter3d(
        x=df_pair.x,
        y=df_pair.y,
        z=df_pair.z,
        mode="markers+lines",
        text=df_pair.label
    )

    data.append(trace)
print(df_pair)

figure = go.Figure(
    data=data,
    layout=go.Layout(

        scene=dict(
            xaxis=dict(title="x", range=(-RANGE, RANGE)),
            yaxis=dict(title="y", range=(-RANGE, RANGE)),
            zaxis=dict(title="z", range=(-RANGE, RANGE))
        ),

    ))

name = ''
# Default parameters which are used when `layout.scene.camera` is not provided
camera = dict(
    up=dict(x=0, y=0, z=1),
    center=dict(x=0, y=0, z=0),
    eye=dict(x=0, y=-1, z=1)
)

figure.update_layout(scene_camera=camera, title=name)  #

figure.show()

      label                                             vector  \
25   Jordan  [0.0041577075608074665, -0.024367135018110275,...   
171   Amman  [-0.006026651244610548, -0.010196387767791748,...   

                                           reduced_vec          x          y  \
25   [-36.85565185546875, -0.17666076123714447, 19.... -36.855652  -0.176661   
171  [-19.898149490356445, -30.829416275024414, 37.... -19.898149 -30.829416   

             z  
25   19.811268  
171  37.525814  


## Przewidywanie stolicy Hiszpanii

In [99]:
# Create country capital vector by averaging several pairs
pairs = [("France", "Paris"), ("Japan", "Tokyo"), ("Italy", "Rome")]
diffs = [word_embeddings[co] - word_embeddings[ca] for co, ca in pairs]
country_capital_vector = np.mean(diffs, axis=0)

Używamy wektoryzacji słów, aby przewidzieć stolicę kraju.

In [100]:
# Improve: Use multiple pairs for a more robust vector
reference_pairs = [("France", "Paris"), ("Japan", "Tokyo"), ("Italy", "Rome")]
diffs = []
for co, ca in reference_pairs:
    v_co = df.query(f"label == '{co}'").vector.values[0]
    v_ca = df.query(f"label == '{ca}'").vector.values[0]
    diffs.append(v_co - v_ca)

country_capital_vector = np.mean(diffs, axis=0)

# Calculate expected vector for Spain's capital in high-D
spain_vec = df.query("label == 'Spain'").vector.values[0]
expected_capital_vector = spain_vec - country_capital_vector

## Wykorzystaj wyszukiwanie najbliższych sąsiadów, aby znaleźć odpowiedni wektor
Wyszukiwanie podobnych sąsiadów w przestrzeni wielowymiarowej. Aby uzyskać większą dokładność, używamy oryginalnych osadzeń.

In [101]:
# Fit nearest neighbors on original 1536D vectors
nearest_neighbors = NearestNeighbors(n_neighbors=3, metric='cosine').fit(full_vectors)

# Find 3 nearest neighbors for expected capital vector
dist, idx = nearest_neighbors.kneighbors(expected_capital_vector.reshape(1, -1))
idx[0]

array([22, 61, 26])

In [102]:
dist

array([[0.05807377, 0.09578421, 0.11946447]])

In [103]:
#Let's see if any of nearest neighbors are correct
df.label.loc[idx[0]]

22     Spain
61    Madrid
26     Paris
Name: label, dtype: str

Utwórz funkcję, która będzie przewidywać stolicę danego kraju.

In [104]:
def get_capital(country, df_data, nn_model, vector_diff):
    """Returns predicted capital for any given country using high-D vector math."""
    try:
        country_vec = df_data.query(f"label == '{country}'").vector.values[0]
    except IndexError:
        return f"Country '{country}' not found in dataset."

    # Compute expected capital vector in high-D
    expected_vec = country_vec - vector_diff

    # Find nearest neighbors
    dist, idx = nn_model.kneighbors(expected_vec.reshape(1, -1))

    # Exclude the country itself from results
    first_neighbor = df_data.iloc[idx[0][0]]
    if first_neighbor.label.lower() == country.lower():
        return df_data.iloc[idx[0][1]]
    return first_neighbor

In [105]:
# Print country capitals for the following list
countries = ["England", "Poland", "Kenya", "Germany"]

for country in countries:
    result = get_capital(country, df, nearest_neighbors, country_capital_vector)
    capital = result.label if not isinstance(result, str) else "Error"
    print(f"{capital} jest przewidywaną stolicą {country}")

London jest przewidywaną stolicą England
Warsaw jest przewidywaną stolicą Poland
Nairobi jest przewidywaną stolicą Kenya
Berlin jest przewidywaną stolicą Germany
